# Load Data

In [31]:
# USE THIS WHEN RUNNING ON COLAB

"""
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df = pd.read_csv(path + '/combined_ipl_mbm_bowl.csv')"""

'\nimport kagglehub\n\n# Download latest version\npath = kagglehub.dataset_download("s1nghayush/data-dump")\n\nprint("Path to dataset files:", path)\n\ndf = pd.read_csv(path + \'/combined_ipl_mbm_bowl.csv\')'

In [32]:
# USE THIS WHEN RUNNING LOCALLY

import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\mbm\combined_ipl_mbm_bowl.csv")

In [33]:
df['date'] = pd.to_datetime(df['date'])

df.sort_values(by=['date', 'inns'], inplace=True)

df['season'] = df['date'].dt.year

df.drop(columns = ['maiden', 'eco', 'lbw_or_bowled', 'dots'], inplace=True)

players_detail = pd.read_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\register_data\player_details.csv")
name_corrector = dict(zip(players_detail['key_cricinfo'], players_detail['cricbuzz_name']))
df['bowler'] = df['bowler_id'].map(name_corrector)

# Career Stats

In [34]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Compute cumulative statistics
df['cumulative_runs'] = df.groupby('bowler_id')['runs'].cumsum() - df['runs']
df['cumulative_wickets'] = df.groupby('bowler_id')['wickets'].cumsum() - df['wickets']
df['cumulative_balls'] = df.groupby('bowler_id')['total_balls'].cumsum() - df['total_balls']
df['cum_matches'] = df.groupby('bowler_id')['bowler_id'].cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute career bowling metrics
df['career_avg'] = df['cumulative_runs'] / df['cumulative_wickets']
df['career_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6).replace(0, 1)
df['career_sr'] = df['cumulative_balls'] / df['cumulative_wickets']
df['career_wpm'] = df['cumulative_wickets'] / df['cum_matches']
df['career_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
career_cols = ['career_avg', 'career_eco', 'career_sr']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in career_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id']).head(1).index, career_cols] = pd.NA

In [44]:
df[df['bowler_id'] == 625383].head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,balls,venue,season,career_avg,career_eco,career_sr,career_wpm,career_opm
21410,2013-04-04,597999,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,1,4.0,32.0,3,24,M Chinnaswamy Stadium,2013,NaN,NaN,NaN,NaN,NaN
22014,2013-04-21,598024,Mumbai Indians,Delhi Capitals,625383,Jasprit Bumrah,2,3.0,38.0,0,21,Arun Jaitley Stadium,2013,10.666667,8.000000,8.0,3.00,4.00
23204,2014-04-19,729287,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,2,2.0,14.0,0,12,Dubai International Cricket Stadium,2014,23.333333,10.000000,14.0,1.50,3.50
23587,2014-05-03,733973,Mumbai Indians,Punjab Kings,625383,Jasprit Bumrah,1,4.0,25.0,0,27,Wankhede Stadium,2014,28.000000,9.333333,18.0,1.00,3.00
23708,2014-05-06,733983,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,2,4.0,22.0,2,24,Wankhede Stadium,2014,36.333333,8.384615,26.0,0.75,3.25


In [45]:
df[df['bowler_id'] == 625383].info() # Career columns can have one row of null values for each bowler

<class 'pandas.core.frame.DataFrame'>
Int64Index: 133 entries, 21410 to 13984
Data columns (total 18 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        133 non-null    datetime64[ns]
 1   match_id    133 non-null    int64         
 2   bowl_team   133 non-null    object        
 3   bat_team    133 non-null    object        
 4   bowler_id   133 non-null    int64         
 5   bowler      133 non-null    object        
 6   inns        133 non-null    int64         
 7   overs       133 non-null    float64       
 8   runs        133 non-null    float64       
 9   wickets     133 non-null    int64         
 10  balls       133 non-null    int64         
 11  venue       133 non-null    object        
 12  season      133 non-null    int64         
 13  career_avg  132 non-null    float64       
 14  career_eco  132 non-null    float64       
 15  career_sr   132 non-null    float64       
 16  career_wpm  132 non-

# Last 5 match stats

In [51]:
# Sort bowling data by bowler and match date
df.sort_values(by=['bowler', 'date'], inplace=True)

# ------------------------------------------------------------------------------
# 1) Bowler Last 5 Match Rolling Bowler's Average
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('bowler_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_wickets = df.groupby('bowler_id')['wickets'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
# Only calculate average when there are wickets taken
df['last5_avg'] = rolling_runs / rolling_wickets
# Replace infinite values with NaN
df['last5_avg'] = df['last5_avg'].replace([np.inf, -np.inf], 0)
df['last5_avg'] = df['last5_avg'].fillna(0)

# ------------------------------------------------------------------------------
# 2) Bowler Last 5 Match Rolling Economy Rate
# ------------------------------------------------------------------------------
rolling_runs = df.groupby('bowler_id')['runs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
rolling_overs = df.groupby('bowler_id')['overs'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
df['last5_eco'] = rolling_runs / rolling_overs.replace(0, 1)

# ------------------------------------------------------------------------------
# 3) Bowler Last 5 Match Rolling Strike Rate (balls per wicket)
# ------------------------------------------------------------------------------
rolling_balls = df.groupby('bowler_id')['balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).sum()
)
# Only calculate strike rate when there are wickets taken
df['last5_sr'] = rolling_balls / rolling_wickets
# Replace infinite values with NaN
df['last5_sr'] = df['last5_sr'].replace([np.inf, -np.inf], 0)
df['last5_sr'] = df['last5_sr'].fillna(0)

# ------------------------------------------------------------------------------
# 4) Bowler Last 5 Match Rolling Wickets per Match
# ------------------------------------------------------------------------------
df['last5_wpm'] = df.groupby('bowler_id')['wickets'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
)

# ------------------------------------------------------------------------------
# 5) Bowler Last 5 Match Rolling Overs per Match
# ------------------------------------------------------------------------------
# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']


df['last5_opm'] = df.groupby('bowler_id')['total_balls'].transform(
    lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()/6
)

df.drop(columns=['overs_int', 'balls_int', 'total_balls'], inplace=True)

last5_cols = ['last5_avg', 'last5_eco', 'last5_sr']
# making the first val as Null
df.loc[df.groupby(['bowler_id']).head(1).index, last5_cols] = pd.NA

In [52]:
df[df['bowler_id'] == 625383].head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,...,career_avg,career_eco,career_sr,career_wpm,career_opm,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm
21410,2013-04-04,597999,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,1,4.0,32.0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22014,2013-04-21,598024,Mumbai Indians,Delhi Capitals,625383,Jasprit Bumrah,2,3.0,38.0,0,...,10.666667,8.000000,8.0,3.00,4.00,10.666667,8.000000,8.0,3.00,4.00
23204,2014-04-19,729287,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,2,2.0,14.0,0,...,23.333333,10.000000,14.0,1.50,3.50,23.333333,10.000000,15.0,1.50,3.50
23587,2014-05-03,733973,Mumbai Indians,Punjab Kings,625383,Jasprit Bumrah,1,4.0,25.0,0,...,28.000000,9.333333,18.0,1.00,3.00,28.000000,9.333333,19.0,1.00,3.00
23708,2014-05-06,733983,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,2,4.0,22.0,2,...,36.333333,8.384615,26.0,0.75,3.25,36.333333,8.384615,28.0,0.75,3.25


In [54]:
df[df['bowler_id'] == 625383].info() # Last 5 match columns can have one row of null values for each bowler

<class 'pandas.core.frame.DataFrame'>
Int64Index: 133 entries, 21410 to 13984
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        133 non-null    datetime64[ns]
 1   match_id    133 non-null    int64         
 2   bowl_team   133 non-null    object        
 3   bat_team    133 non-null    object        
 4   bowler_id   133 non-null    int64         
 5   bowler      133 non-null    object        
 6   inns        133 non-null    int64         
 7   overs       133 non-null    float64       
 8   runs        133 non-null    float64       
 9   wickets     133 non-null    int64         
 10  balls       133 non-null    int64         
 11  venue       133 non-null    object        
 12  season      133 non-null    int64         
 13  career_avg  132 non-null    float64       
 14  career_eco  132 non-null    float64       
 15  career_sr   132 non-null    float64       
 16  career_wpm  132 non-

# Venue stats

In [55]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Group by bowler and venue to compute cumulative statistics
grouped = df.groupby(['bowler_id', 'venue'])

# Compute cumulative statistics
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']
df['cumulative_wickets'] = grouped['wickets'].cumsum() - df['wickets']
df['cumulative_balls'] = grouped['total_balls'].cumsum() - df['total_balls']
df['cum_matches'] = grouped.cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute venue-wise bowling metrics
df['venue_avg'] = df['cumulative_runs'] / df['cumulative_wickets']
df['venue_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6)
df['venue_sr'] = df['cumulative_balls'] / df['cumulative_wickets']
df['venue_wpm'] = df['cumulative_wickets'] / df['cum_matches']
df['venue_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
venue_cols = ['venue_avg', 'venue_eco', 'venue_sr']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in venue_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'venue']).head(1).index, venue_cols] = pd.NA

In [58]:
df[(df['bowler_id'] == 625383) & (df['venue'] == 'Wankhede Stadium')].head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,...,last5_avg,last5_eco,last5_sr,last5_wpm,last5_opm,venue_avg,venue_eco,venue_sr,venue_wpm,venue_opm
23587,2014-05-03,733973,Mumbai Indians,Punjab Kings,625383,Jasprit Bumrah,1,4.0,25.0,0,...,28.000000,9.333333,19.0,1.00,3.00,NaN,NaN,NaN,NaN,NaN
23708,2014-05-06,733983,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,2,4.0,22.0,2,...,36.333333,8.384615,28.0,0.75,3.25,0.0,6.250000,0.0,0.000000,4.0
23841,2014-05-10,733995,Mumbai Indians,Chennai Super Kings,625383,Jasprit Bumrah,2,4.0,33.0,0,...,26.200000,7.705882,21.6,1.00,3.40,23.5,5.875000,24.0,1.000000,4.0
24235,2014-05-23,734031,Mumbai Indians,Delhi Capitals,625383,Jasprit Bumrah,2,4.0,36.0,0,...,67.000000,7.444444,55.5,0.40,3.60,40.0,6.666667,36.0,0.666667,4.0
24335,2014-05-25,734041,Mumbai Indians,Rajasthan Royals,625383,Jasprit Bumrah,1,4.0,30.0,1,...,68.500000,7.611111,55.0,0.40,3.60,58.0,7.250000,48.0,0.500000,4.0


In [59]:
print("No of unique venues played at by Jasprit Bumrah : ",len(df[(df['bowler_id'] == 625383)]['venue'].unique()))
df[(df['bowler_id'] == 625383)].info() # NOTE - Venue stats can have as many row of null values for each bowler as the number of venues played at

No of unique venues played at by Jasprit Bumrah :  22
<class 'pandas.core.frame.DataFrame'>
Int64Index: 133 entries, 21410 to 13984
Data columns (total 28 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        133 non-null    datetime64[ns]
 1   match_id    133 non-null    int64         
 2   bowl_team   133 non-null    object        
 3   bat_team    133 non-null    object        
 4   bowler_id   133 non-null    int64         
 5   bowler      133 non-null    object        
 6   inns        133 non-null    int64         
 7   overs       133 non-null    float64       
 8   runs        133 non-null    float64       
 9   wickets     133 non-null    int64         
 10  balls       133 non-null    int64         
 11  venue       133 non-null    object        
 12  season      133 non-null    int64         
 13  career_avg  132 non-null    float64       
 14  career_eco  132 non-null    float64       
 15  career_sr   13

# Innings Stats

In [62]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Group by bowler and innings
grouped = df.groupby(['bowler_id', 'inns'])

# Compute cumulative statistics for each bowler in each innings
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']  # Subtract current row's runs
df['cumulative_wickets'] = grouped['wickets'].cumsum() - df['wickets']  # Subtract current row's wickets
df['cumulative_balls'] = grouped['total_balls'].cumsum() - df['total_balls']  # Subtract current row's balls
df['cum_matches'] = grouped.cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute innings-wise bowling metrics
df['inns_avg'] = df['cumulative_runs'] / df['cumulative_wickets']  # Average runs per wicket
df['inns_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6).replace(0,1)  # Economy rate
df['inns_sr'] = df['cumulative_balls'] / df['cumulative_wickets']  # Strike rate (balls per wicket)
df['inns_wpm'] = df['cumulative_wickets'] / df['cum_matches']  # Wickets per match
df['inns_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']  # Overs per match

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
inns_cols = ['inns_avg', 'inns_sr', 'inns_eco']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in inns_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'inns']).head(1).index, inns_cols] = pd.NA

In [63]:
df[(df['bowler_id'] == 625383) & (df['inns'] == 1)].head()

,date,match_id,bowl_team,bat_team,bowler_id,bowler,inns,overs,runs,wickets,...,venue_avg,venue_eco,venue_sr,venue_wpm,venue_opm,inns_avg,inns_eco,inns_sr,inns_wpm,inns_opm
21410,2013-04-04,597999,Mumbai Indians,Royal Challengers Bangalore,625383,Jasprit Bumrah,1,4.0,32.0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23587,2014-05-03,733973,Mumbai Indians,Punjab Kings,625383,Jasprit Bumrah,1,4.0,25.0,0,...,NaN,NaN,NaN,NaN,NaN,10.666667,8.000,8.0,3.00,4.0
23895,2014-05-12,734001,Mumbai Indians,Sunrisers Hyderabad,625383,Jasprit Bumrah,1,4.0,24.0,0,...,NaN,NaN,NaN,NaN,NaN,19.000000,7.125,16.0,1.50,4.0
24160,2014-05-21,734025,Mumbai Indians,Punjab Kings,625383,Jasprit Bumrah,1,4.0,31.0,2,...,NaN,NaN,NaN,NaN,NaN,27.000000,6.750,24.0,1.00,4.0
24335,2014-05-25,734041,Mumbai Indians,Rajasthan Royals,625383,Jasprit Bumrah,1,4.0,30.0,1,...,58.0,7.25,48.0,0.5,4.0,22.400000,7.000,19.2,1.25,4.0


In [64]:
df[(df['bowler_id'] == 625383)].info() # Innings stats can have two rows of null values for each bowler

<class 'pandas.core.frame.DataFrame'>
Int64Index: 133 entries, 21410 to 13984
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        133 non-null    datetime64[ns]
 1   match_id    133 non-null    int64         
 2   bowl_team   133 non-null    object        
 3   bat_team    133 non-null    object        
 4   bowler_id   133 non-null    int64         
 5   bowler      133 non-null    object        
 6   inns        133 non-null    int64         
 7   overs       133 non-null    float64       
 8   runs        133 non-null    float64       
 9   wickets     133 non-null    int64         
 10  balls       133 non-null    int64         
 11  venue       133 non-null    object        
 12  season      133 non-null    int64         
 13  career_avg  132 non-null    float64       
 14  career_eco  132 non-null    float64       
 15  career_sr   132 non-null    float64       
 16  career_wpm  132 non-

# Season Stats

In [65]:
# Convert 'DNB' to 0 in overs column
df['overs'] = df['overs'].replace('DNB', '0.0')

# Convert overs column to numeric format (split into overs and balls)
df[['overs_int', 'balls_int']] = df['overs'].astype(str).str.split('.', expand=True).astype(int)

# Convert to total balls for correct summation
df['total_balls'] = df['overs_int'] * 6 + df['balls_int']

# Group by bowler and season
grouped = df.groupby(['bowler_id', 'season'])

# Compute cumulative statistics for each bowler in each season
df['cumulative_runs'] = grouped['runs'].cumsum() - df['runs']  # Subtract current row's runs
df['cumulative_wickets'] = grouped['wickets'].cumsum() - df['wickets']  # Subtract current row's wickets
df['cumulative_balls'] = grouped['total_balls'].cumsum() - df['total_balls']  # Subtract current row's balls
df['cum_matches'] = grouped.cumcount()

# Convert total balls back to overs format
df['cumulative_overs'] = (df['cumulative_balls'] // 6).astype(str) + "." + (df['cumulative_balls'] % 6).astype(str)

# Compute season-wise bowling metrics
df['ssn_avg'] = df['cumulative_runs'] / df['cumulative_wickets'] # Average runs per wicket
df['ssn_eco'] = df['cumulative_runs'] / (df['cumulative_balls'] / 6)  # Economy rate
df['ssn_sr'] = df['cumulative_balls'] / df['cumulative_wickets']  # Strike rate (balls per wicket)
df['ssn_wpm'] = df['cumulative_wickets'] / df['cum_matches']  # Wickets per match
df['ssn_opm'] = (df['cumulative_balls'] / 6) / df['cum_matches']  # Overs per match

# Drop temporary columns
df.drop(columns=['overs_int', 'balls_int', 'total_balls', 'cumulative_runs',	'cumulative_wickets',	'cumulative_balls',	'cum_matches',	'cumulative_overs'], inplace=True)

# Columns to process
ssn_cols = ['ssn_avg', 'ssn_sr', 'ssn_eco']

# Replace inf/-inf with 0 and Nan with 0 in the specified columns
for col in ssn_cols:
  df[col] = df[col].replace([np.inf, -np.inf], 0.0)
  df[col] = df[col].fillna(0)

# making the first val as Null
df.loc[df.groupby(['bowler_id', 'season']).head(1).index, ssn_cols] = pd.NA

In [66]:
print("No of unique seasons played by Jasprit Bumrah : ",len(df[(df['bowler_id'] == 625383)]['season'].unique()))
df[(df['bowler_id'] == 625383)].info() # NOTE - Season stats can have as many row of null values for each bowler as the number of seasons played

No of unique seasons played by Jasprit Bumrah :  11
<class 'pandas.core.frame.DataFrame'>
Int64Index: 133 entries, 21410 to 13984
Data columns (total 38 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        133 non-null    datetime64[ns]
 1   match_id    133 non-null    int64         
 2   bowl_team   133 non-null    object        
 3   bat_team    133 non-null    object        
 4   bowler_id   133 non-null    int64         
 5   bowler      133 non-null    object        
 6   inns        133 non-null    int64         
 7   overs       133 non-null    float64       
 8   runs        133 non-null    float64       
 9   wickets     133 non-null    int64         
 10  balls       133 non-null    int64         
 11  venue       133 non-null    object        
 12  season      133 non-null    int64         
 13  career_avg  132 non-null    float64       
 14  career_eco  132 non-null    float64       
 15  career_sr   132 

# Cleaning and Exporting

In [68]:
df.drop(columns=['overs', 'runs', 'wickets', 'balls'], inplace=True)

In [71]:
df = df.reindex(columns=['date', 'match_id', 'bowl_team', 'bat_team', 'bowler_id', 'bowler', 'inns', 'venue', 'season',
                         'career_avg', 'career_eco', 'career_sr', 'career_wpm', 'career_opm',
                         'inns_avg', 'inns_eco', 'inns_sr', 'inns_wpm', 'inns_opm',
                         'venue_avg', 'venue_eco', 'venue_sr', 'venue_wpm', 'venue_opm',
                         'ssn_avg', 'ssn_eco', 'ssn_sr', 'ssn_wpm', 'ssn_opm',
                         'last5_avg', 'last5_eco', 'last5_sr', 'last5_wpm', 'last5_opm'])

In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24444 entries, 14694 to 3795
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        24444 non-null  datetime64[ns]
 1   match_id    24444 non-null  int64         
 2   bowl_team   24444 non-null  object        
 3   bat_team    24444 non-null  object        
 4   bowler_id   24444 non-null  int64         
 5   bowler      24444 non-null  object        
 6   inns        24444 non-null  int64         
 7   venue       24444 non-null  object        
 8   season      24444 non-null  int64         
 9   career_avg  23709 non-null  float64       
 10  career_eco  23709 non-null  float64       
 11  career_sr   23709 non-null  float64       
 12  career_wpm  23709 non-null  float64       
 13  career_opm  23709 non-null  float64       
 14  inns_avg    22967 non-null  float64       
 15  inns_eco    22967 non-null  float64       
 16  inns_sr     22967 n

In [73]:
filtered_df = df[df['date'] > '2015-01-01']

In [74]:
filtered_df['date'].min(), filtered_df['date'].max()

(Timestamp('2015-04-08 00:00:00'), Timestamp('2024-05-26 00:00:00'))

In [75]:
df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\ipl_bowl_features.csv", index=False)